In [12]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
from datetime import datetime

os.environ["AWS_REGION"] = "us-east-2"

spark = SparkSession.builder \
    .appName("test-silver") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

BUCKET = "clashr-account-data-lake"
BATTLE_PATH = f"s3a://{BUCKET}/raw/battles/"
PROFILE_PATH = f"s3a://{BUCKET}/raw/profile/"

df_battles_raw = (
    spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(BATTLE_PATH)
)

df_profile_raw = (
    spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(PROFILE_PATH)
)

print("✅ Dados da Bronze carregados")

✅ Dados da Bronze carregados


In [13]:
df_battles = df_battles_raw.select(
    F.col("player_tag"),
    F.col("battle_time"),
    F.col("battle_type"),
    F.col("result"),
    F.col("player_trophies"),
    F.col("player_deck"),
    F.col("player_elixir_avg").cast("float"),
    F.col("crowns_player").cast("int"),
    F.col("crowns_opponent").cast("int"),
    F.col("opponent_tag"),
    F.col("opponent_deck"),
)


# Converte timestamp
df_battles = df_battles.withColumn(
    "battle_time",
    F.to_timestamp(F.col("battle_time"), "yyyyMMdd'T'HHmmss.SSS'Z'")
)

# Adiciona partição e IDs
df_battles = df_battles.withColumn("ingestion_date", F.current_date())
df_battles = df_battles.withColumn(
    "battle_id",
    F.md5(F.concat_ws("_", F.col("player_tag"), F.col("battle_time").cast("string")))
)

df_battles = df_battles.withColumn(
    "deck_hash",
    F.md5(F.array_join(F.sort_array(F.col("player_deck")), "|"))
)

# Limpeza
df_battles_clean = df_battles \
    .dropna(subset=["player_tag", "battle_time", "result"]) \
    .dropDuplicates(["battle_id"])

df_battles.show()

+----------+-------------------+-----------+------+---------------+--------------------+-----------------+-------------+---------------+------------+--------------------+--------------+--------------------+--------------------+
|player_tag|        battle_time|battle_type|result|player_trophies|         player_deck|player_elixir_avg|crowns_player|crowns_opponent|opponent_tag|       opponent_deck|ingestion_date|           battle_id|           deck_hash|
+----------+-------------------+-----------+------+---------------+--------------------+-----------------+-------------+---------------+------------+--------------------+--------------+--------------------+--------------------+
| 8Q8UQPP8L|2026-06-03 00:28:44|      trail|  loss|              0|[Hunter, Goblin D...|             0.84|            0|              1|   8RGUUQVLR|[Rocket, Royal De...|    2026-06-11|6b5f57287889bfc49...|04912ee8fe4c6b768...|
| 8Q8UQPP8L|2026-06-02 19:55:08|        PvP|   win|          11180|[Dart Goblin, Kni...|

In [ ]:
# Valida os dados transformados

print("\n=== SCHEMA SILVER BATTLES ===")
df_battles_clean.printSchema()

print("\n=== AMOSTRA DOS DADOS TRANSFORMADOS ===")
df_battles_clean.select(
    "player_tag", "battle_time", "result",
    "crowns_player", "crowns_opponent", "battle_id"
).show(5)

print("\n=== VALIDAÇÕES ===")
print(f"Nulos em player_tag: {df_battles_clean.filter(F.col('player_tag').isNull()).count()}")
print(f"Nulos em battle_time: {df_battles_clean.filter(F.col('battle_time').isNull()).count()}")
print(f"Nulos em battle_id: {df_battles_clean.filter(F.col('battle_id').isNull()).count()}")
print(f"Battle IDs únicos: {df_battles_clean.select('battle_id').distinct().count()}")

print("\n=== DISTRIBUIÇÃO DE RESULTADOS ===")
df_battles_clean.groupBy("result").count().show()

print("\n=== DISTRIBUIÇÃO DE DECK_HASH ===")
print(f"Total de decks únicos: {df_battles_clean.select('deck_hash').distinct().count()}")

print("\n✅ Transformação battles validada!")


In [14]:
# Testa transformação de perfil

print("\n=== TRANSFORMANDO PERFIL BRONZE → SILVER ===\n")

df_profile = df_profile_raw.select(
    F.col("tag").alias("player_tag"),
    F.col("name").alias("player_name"),
    F.col("level").cast("int"),
    F.col("trophies").cast("int"),
    F.col("best_trophies").cast("int"),
    F.col("wins").cast("int"),
    F.col("losses").cast("int"),
    F.col("battle_count").cast("int"),
    F.col("clan_name"),
    F.col("clan_tag"),
)

# Calcula win rate
df_profile = df_profile.withColumn(
    "win_rate",
    F.when(
        F.col("battle_count") > 0,
        F.round(F.col("wins") / F.col("battle_count") * 100, 2)
    ).otherwise(0.0)
)

df_profile = df_profile \
    .withColumn("snapshot_at", F.current_timestamp()) \
    .withColumn("ingestion_date", F.current_date())

df_profile_clean = df_profile \
    .dropna(subset=["player_tag", "player_name"]) \
    .dropDuplicates(["player_tag", "snapshot_at"])

print(f"Registros: {df_profile_clean.count()}")

print("\n=== AMOSTRA PERFIL TRANSFORMADO ===")
df_profile_clean.select(
    "player_tag", "player_name", "trophies", "win_rate", "clan_name"
).show(3)

print("\n=== WIN RATE STATS ===")
df_profile_clean.agg(
    F.min("win_rate").alias("min_win_rate"),
    F.max("win_rate").alias("max_win_rate"),
    F.round(F.avg("win_rate"), 2).alias("avg_win_rate")
).show()

print("\n✅ Transformação profile validada!")


=== TRANSFORMANDO PERFIL BRONZE → SILVER ===



Registros: 1

=== AMOSTRA PERFIL TRANSFORMADO ===


+----------+-----------+--------+--------+----------+
|player_tag|player_name|trophies|win_rate| clan_name|
+----------+-----------+--------+--------+----------+
| 8Q8UQPP8L|   MingaL04|   11210|   52.85|FLUMINENSE|
+----------+-----------+--------+--------+----------+


=== WIN RATE STATS ===
+------------+------------+------------+
|min_win_rate|max_win_rate|avg_win_rate|
+------------+------------+------------+
|       52.85|       52.85|       52.85|
+------------+------------+------------+


✅ Transformação profile validada!
